# Real-World Data Project – Health
## Diabetes Disease Progression Analysis & Prediction

### Objective
Analyze a real-world health dataset and build regression models to predict a quantitative measure of disease progression one year after baseline.

### Project Workflow
**Load Data → Explore → Clean → Visualize → Correlation Analysis → Train Models → Evaluate → Compare → Generate Insights**

### Important Dataset Note
The target variable represents a quantitative measure of disease progression one year after baseline. It is **not a diagnosis of diabetes** and should not be interpreted as a medical diagnosis.

### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Jupyter Notebook


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme()
plt.rcParams["figure.figsize"] = (9, 5)


## 2. Load the Dataset

In [ ]:
# The CSV is included in this project folder.
df = pd.read_csv("diabetes_real_world_dataset.csv")
df.head()


## 3. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())


## 4. Data Quality Check

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe())


### Data Quality Result
The dataset is already numeric and structured for analysis. Missing values and duplicate records are checked before modeling. Because the data is compact and standardized, no unnecessary transformations are applied.


## 5. Target Variable Distribution

In [ ]:
plt.figure()
plt.hist(df["target"], bins=25)
plt.title("Distribution of Disease Progression Target")
plt.xlabel("Target value")
plt.ylabel("Frequency")
plt.show()


## 6. Feature Distributions

In [ ]:
df.drop(columns="target").hist(figsize=(12, 10), bins=20)
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()


## 7. Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


## 8. Strongest Correlations with the Target

In [ ]:
target_corr = corr["target"].drop("target").sort_values(key=abs, ascending=False)
display(target_corr.to_frame("correlation_with_target"))


## 9. Most Related Feature vs Target

In [ ]:
strongest_feature = target_corr.index[0]
print("Feature with the strongest absolute correlation:", strongest_feature)

plt.figure()
sns.scatterplot(data=df, x=strongest_feature, y="target")
plt.title(f"{strongest_feature} vs Target")
plt.xlabel(strongest_feature)
plt.ylabel("Target")
plt.show()


## 10. Prepare Data for Machine Learning

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 11. Model 1 – Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("MAE :", round(linear_mae, 2))
print("RMSE:", round(linear_rmse, 2))
print("R²  :", round(linear_r2, 2))


## 12. Model 2 – Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Regression")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 2))


## 13. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [linear_mae, rf_mae],
    "RMSE": [linear_rmse, rf_rmse],
    "R2": [linear_r2, rf_r2]
})

display(results.round(3))


## 14. Actual vs Predicted Values

In [ ]:
plt.figure()
plt.scatter(y_test, linear_pred, alpha=0.7, label="Linear Regression")
plt.scatter(y_test, rf_pred, alpha=0.7, label="Random Forest")

minimum = min(y_test.min(), linear_pred.min(), rf_pred.min())
maximum = max(y_test.max(), linear_pred.max(), rf_pred.max())
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")

plt.title("Actual vs Predicted Target")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend()
plt.show()


## 15. Random Forest Feature Importance

In [ ]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importance.to_frame("importance"))

plt.figure()
importance.sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## 16. Residual Analysis

In [ ]:
residuals = y_test - rf_pred

plt.figure()
plt.scatter(rf_pred, residuals)
plt.axhline(0, linestyle="--")
plt.title("Random Forest Residual Analysis")
plt.xlabel("Predicted Value")
plt.ylabel("Residual")
plt.show()


## 17. Key Findings

In [ ]:
print("Dataset size:", df.shape)
print("Number of features:", X.shape[1])
print("Strongest correlation with target:", strongest_feature)
print("\nModel comparison:")
display(results.round(3))

print("\nRandom Forest feature importance:")
display(importance.to_frame("importance").head())


## 18. Conclusion

This project demonstrates an end-to-end real-world health data workflow.

### Main conclusions
- The dataset was explored using descriptive statistics and visualizations.
- Data quality was checked for missing values and duplicate records.
- Correlation analysis was used to study relationships between health-related features and the target.
- Two regression approaches were implemented: Linear Regression and Random Forest Regression.
- MAE, RMSE, and R² were used to evaluate model performance.
- Feature importance from Random Forest provides an additional view of which variables contribute most to model predictions.

### Practical Use
A workflow like this can help researchers and analysts understand patterns in health datasets and build predictive models for further study.

### Limitation
This dataset is relatively small and the target is a quantitative disease-progression measure. Model predictions should not be treated as clinical advice or a medical diagnosis.


## 19. References

1. Efron, B., Hastie, T., Johnstone, I., & Tibshirani, R. (2004). Least Angle Regression.
2. Scikit-learn documentation – Diabetes dataset and regression models.
3. Scikit-learn `load_diabetes` dataset, a standard real-world regression dataset used for machine-learning education and research.

**Dataset source:** Scikit-learn built-in Diabetes dataset.
